In [1]:
import cdsapi
import xarray as xr
import pandas as pd
import numpy as np

import geogridfusion
from fips import fips_to_state_county, state_fips_mapping
from sodapy import Socrata

In [2]:
conn = geogridfusion.start()

Starting Postgres subprocess...
PostgreSQL connection established after 4.91 seconds.
postgis already installed


In [3]:
geogridfusion.sources(conn=conn)

{'era5-reanalysis': 401}

Map of soiling sites in the US  
https://www2.nrel.gov/pv/soiling

### Data sources

ERA5 data for rain
https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=download

Particulate 2.5 nm
https://data.cdc.gov/Environmental-Health-Toxicology/Daily-County-Level-PM2-5-Concentrations-2001-2019/dqwm-pbi7/data_preview

In [ ]:
### era 5 rain download
dataset = "reanalysis-era5-single-levels"
request = {
    "product_type": ["reanalysis"],
    "variable": ["total_precipitation"],
    "year": ["2016"],
    "month": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"
    ],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "time": [
        "00:00", "01:00", "02:00",
        "03:00", "04:00", "05:00",
        "06:00", "07:00", "08:00",
        "09:00", "10:00", "11:00",
        "12:00", "13:00", "14:00",
        "15:00", "16:00", "17:00",
        "18:00", "19:00", "20:00",
        "21:00", "22:00", "23:00"
    ],
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": [49.95, -125, 24.9, -66],
    "grid":[1,1],
}

client = cdsapi.Client()
client.retrieve(dataset, request).download()

In [ ]:
loaded = xr.open_dataset("354e32a51aadddd3597b21c7cfa43963.nc", engine='netcdf4')
loaded = loaded.drop_vars(["number", "expver"]).rename({"valid_time":"time"})
loaded

In [ ]:
import multiprocessing as mp
from functools import partial


    
def ingest_dataset_parallel(dataset: xr.Dataset, skip_null:bool=True, include_data_vars:list[str]=None)
    """
    ingest a geospatial dataset in parallel.
    
    """
    tasks = []

    if not all(["latitude", "longitude"] in list(dataset.coords)):
        raise ValueError(f"latitude and longitude expected in dataset coords, existing dataset coords: {dataset.coords}")

    for i, lat in enumerate(loaded.latitude.values):
        for j, lon in enumerate(loaded.longitude.values):
            tasks.append((i, j, lat, lon))


In [ ]:
for i, lat in enumerate(loaded.latitude.values):
    for j, lon in enumerate(loaded.longitude.values):

            single_data = loaded.isel(latitude=i, longitude=j).drop(["latitude","longitude"])

            if any(loaded.isel(latitude=0, longitude=0).drop_vars(["latitude","longitude"]).tp.isnull()):
                print(f"lat: {lat}, lon: {lon} has null entries, skipping")
                continue

            ingest_weather = single_data.to_dataframe()
            ingest_meta = {
                "Source":"ERA5-reanalysis",
                "latitude":lat,
                "longitude":lon,
                "year":2016
            }

            geogridfusion.store_single(
                conn=conn,
                weather_df=ingest_weather,
                meta=ingest_meta, 
                tmy=False,
                source_name=ingest_meta["Source"]
            )

In [ ]:
geogridfusion.sources(conn=conn)

In [ ]:
precip_data, precip_meta = geogridfusion.load_many(conn=conn, source_name="era5-reanalysis")

In [ ]:
precip_data

In [ ]:
loaded.tp.mean(dim="valid_time").plot()

In [ ]:
def read_dataset_convert_to_geospatial_xarray() -> list[xr.Dataset]:
    states_ds = []    

    client = Socrata("data.cdc.gov", None)
    dataaset_identifier = "dqwm-pbi7"

    for statefips in state_fips_mapping.keys():
        
        if statefips in ['02', '15']: # skip AK, HI
            continue

        results = client.get(
            dataset_identifier=dataaset_identifier,
            limit=150000,
            select="year, date, statefips, countyfips, PM25_mean_pred",
            where=f"statefips={statefips} AND year >= '2016' AND year < '2017' ORDER BY statefips, countyfips, date"
        )

        # load query from dataset
        df = pd.DataFrame(results)
        df["datetime"] = pd.to_datetime(df["date"], format="%d%b%Y", errors="coerce")
        df = df.set_index("datetime")

        df = df.drop(columns=["year", 'date'])
        df = df.astype({"statefips": int, 'countyfips':int, "PM25_mean_pred":float})

        # get state, county names using mapping
        
        df[['state_name', 'county_name']] = df.apply(
            lambda row: pd.Series(fips_to_state_county(
                state_fips=int(row['statefips']), 
                county_fips=int(row['countyfips']))
            ),
            axis=1
        )

        # load fips-> coords mapping (sourced from kaggle)
        counties_centroids = pd.read_csv("County Centroids.csv",index_col=[0,1])
        counties_centroids.index = pd.MultiIndex.from_tuples([(str(a).lower(), str(b).lower()) for a, b in counties_centroids.index])

        def safe_lookup(row):
            try:
                return counties_centroids.loc[
                    (row["state_name"].lower(), row["county_name"].lower()),
                    ["latitude", "longitude"]
                ]
            except Exception:
                return pd.Series([None, None], index=["latitude", "longitude"])
        lookup_result = df.apply(safe_lookup, axis=1)

        # Keep only successful lookups
        valid = lookup_result.notnull()
        df_valid = df[valid]
        df_valid[["latitude", "longitude"]] = lookup_result[valid].values

        # drop extra columns
        df = df_valid.drop(columns=["statefips","countyfips"])
        df= df.reset_index() # expose "datetime" column

        lats = np.sort(np.unique(df.latitude.values))
        lons = np.sort(np.unique(df.longitude.values))
        times = pd.date_range(start="2016-01-01", freq="1d", periods=366)

        protype_ds = xr.Dataset(
            data_vars={"pm25": (("latitude","longitude","time"), np.empty(shape=(len(lats), len(lons), 366)))},
            coords={
                "latitude": lats,
                "longitude": lons,
                "time": times
            }
        )

        lat_idx = {lat: i for i, lat in enumerate(lats)}
        lon_idx = {lon: i for i, lon in enumerate(lons)}
        time_idx = {time: i for i, time in enumerate(times)}

        # populate dataset
        for _, row in df.iterrows():
            lat = row["latitude"]
            lon = row["longitude"]
            time = row["datetime"]
            val = row["PM25_mean_pred"]

            # Skip rows outside the defined time range
            if (lat not in lat_idx) or (lon not in lon_idx) or (time not in time_idx):
                continue

            try:
                i = lat_idx[lat]
                j = lon_idx[lon]
                k = time_idx[time]
            except KeyError:
                continue

            protype_ds.pm25.values[i, j, k] = val

        # states_ds.append(protype_ds)
        protype_ds.to_netcdf(f"states_pm/fips-{statefips}.nc")

In [ ]:
statefips = "08"
client = Socrata("data.cdc.gov", None)
dataaset_identifier = "dqwm-pbi7"


results = client.get(
        dataset_identifier=dataaset_identifier,
        limit=150000,
        select="year, date, statefips, countyfips, PM25_mean_pred",
        where=f"statefips={statefips} AND year >= '2016' AND year < '2017' ORDER BY statefips, countyfips, date"
    )

# load query from dataset
df = pd.DataFrame(results)
df["datetime"] = pd.to_datetime(df["date"], format="%d%b%Y", errors="coerce")
df = df.set_index("datetime")

df = df.drop(columns=["year", 'date'])
df = df.astype({"statefips": int, 'countyfips':int, "PM25_mean_pred":float})

# get state, county names using mapping

df[['state_name', 'county_name']] = df.apply(
    lambda row: pd.Series(fips_to_state_county(
        state_fips=int(row['statefips']), 
        county_fips=int(row['countyfips']))
    ),
    axis=1
)

# load fips-> coords mapping (sourced from kaggle)
counties_centroids = pd.read_csv("County Centroids.csv",index_col=[0,1])
counties_centroids.index = pd.MultiIndex.from_tuples([(str(a).lower(), str(b).lower()) for a, b in counties_centroids.index])

df[["latitude","longitude"]] = df.apply(axis=1, func=lambda row: counties_centroids.loc[(row["state_name"].lower(), row["county_name"].lower())])[["latitude", "longitude"]]

df = df.drop(columns=["statefips","countyfips"])
df= df.reset_index() # expose "datetime" column

grouped = df.groupby(["latitude", "longitude"])

# Create a dictionary of DataFrames
split_dfs = {
    (state, county): group.copy()
    for (state, county), group in grouped
}


time_ordered_split_dfs = {
    (float(coords[0]), float(coords[1])) : df.sort_index() for coords, df in split_dfs.items()
}

for coords, final_df in time_ordered_split_dfs.items(): 
    print(f"saving coords:{coords}")

    meta = {'year': 2016} | final_df.iloc[0][["state_name", "county_name", "latitude", "longitude"]].to_dict()
    weather = final_df.drop(columns=["state_name", "county_name"])

    geogridfusion.store_single(
        conn=conn,
        weather_df=weather,
        meta=meta,
        source_name="cdc-daily-particulate-forecasts",
        tmy=False,
    )

In [ ]:
list(time_ordered_split_dfs.values())[0]

In [ ]:
wkklist(split_dfs.values())[63].sort_values(by="datetime")

In [ ]:
res = read_dataset_convert_to_geospatial_xarray()

In [ ]:
xr.open_dataset("states_pm/fips-01.nc").pm25